# 🔮 Comparaison des modèles de prévision
## Projet P9 — Librairie Lapage

---

Ce notebook compare plusieurs approches de prévision pour trouver le meilleur modèle.

### Modèles testés

| Modèle | Description |
|--------|-------------|
| Holt-Winters (optimisé) | Lissage exponentiel avec tuning |
| SARIMA | Auto-ARIMA avec saisonnalité |
| Régression + Features | Ridge/Random Forest avec features temporelles |
| Moyenne mobile | Baseline simple |

In [1]:
# Imports
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Statsmodels
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose

# ML
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Visualisation
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("✅ Imports réussis")

✅ Imports réussis


---
## 1. Chargement et préparation

In [2]:
# Charger les transactions
df = pd.read_csv('../data/processed/transactions_enrichies.csv')
df['date'] = pd.to_datetime(df['date'])

# Agrégation par jour
df_daily = df.groupby(pd.Grouper(key='date', freq='D')).agg(ca=('price', 'sum')).reset_index()

# Index datetime complet (combler les trous)
date_range = pd.date_range(start=df_daily['date'].min(), end=df_daily['date'].max(), freq='D')
df_daily = df_daily.set_index('date').reindex(date_range).fillna(0)
df_daily.index.name = 'date'
df_daily = df_daily.reset_index()

print(f"📊 {len(df_daily)} jours de données")
print(f"📅 Période : {df_daily['date'].min().date()} → {df_daily['date'].max().date()}")

📊 730 jours de données
📅 Période : 2021-03-01 → 2023-02-28


In [3]:
# Split Train/Test (30 derniers jours pour évaluation)
HORIZON = 30

train = df_daily.iloc[:-HORIZON].copy()
test = df_daily.iloc[-HORIZON:].copy()

print(f"Train : {len(train)} jours")
print(f"Test  : {len(test)} jours")

Train : 700 jours
Test  : 30 jours


In [4]:
# Fonction d'évaluation
def evaluate_model(y_true, y_pred, model_name):
    """Calcule et affiche les métriques."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    # MAPE en évitant division par zéro
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    
    return {
        'model': model_name,
        'MAE': round(mae, 2),
        'RMSE': round(rmse, 2),
        'MAPE': round(mape, 2)
    }

# Stocker les résultats
results = []

---
## 2. Baseline : Moyenne mobile

In [5]:
# Moyenne mobile sur les 7 derniers jours
window = 7
last_values = train['ca'].tail(window).values
pred_baseline = np.full(HORIZON, last_values.mean())

# Évaluation
results.append(evaluate_model(test['ca'].values, pred_baseline, 'Baseline (MM7)'))
print(f"✅ Baseline : MAE = {results[-1]['MAE']:.2f}€")

✅ Baseline : MAE = 1276.03€


---
## 3. Holt-Winters optimisé

In [6]:
# Tester différentes configurations
hw_configs = [
    {'trend': 'add', 'seasonal': 'add', 'damped': False, 'name': 'HW Add/Add'},
    {'trend': 'add', 'seasonal': 'add', 'damped': True, 'name': 'HW Add/Add Damped'},
    {'trend': 'add', 'seasonal': 'mul', 'damped': False, 'name': 'HW Add/Mul'},
    {'trend': 'add', 'seasonal': 'mul', 'damped': True, 'name': 'HW Add/Mul Damped'},
    {'trend': 'mul', 'seasonal': 'mul', 'damped': True, 'name': 'HW Mul/Mul Damped'},
]

ts_train = train.set_index('date')['ca']
best_hw_mae = float('inf')
best_hw_pred = None
best_hw_name = None

print("🔧 Test des configurations Holt-Winters...")

for config in hw_configs:
    try:
        model = ExponentialSmoothing(
            ts_train,
            trend=config['trend'],
            seasonal=config['seasonal'],
            seasonal_periods=7,
            damped_trend=config['damped']
        )
        fit = model.fit(optimized=True)
        pred = fit.forecast(HORIZON)
        
        mae = mean_absolute_error(test['ca'].values, pred.values)
        print(f"   {config['name']:25} → MAE = {mae:,.2f}€")
        
        if mae < best_hw_mae:
            best_hw_mae = mae
            best_hw_pred = pred.values
            best_hw_name = config['name']
            best_hw_model = fit
    except Exception as e:
        print(f"   {config['name']:25} → ❌ Erreur: {str(e)[:30]}")

print(f"\n✅ Meilleur HW : {best_hw_name} (MAE = {best_hw_mae:,.2f}€)")
results.append(evaluate_model(test['ca'].values, best_hw_pred, f'Holt-Winters ({best_hw_name})'))

🔧 Test des configurations Holt-Winters...
   HW Add/Add                → MAE = 1,132.98€
   HW Add/Add Damped         → MAE = 1,119.02€


c:\Users\RémiJulien\OneDrive\Documents\DcidConsulting\2.Prestation\3.Formation\3.Production\2025_GRETA DEV IA_P5\OCR_Projet_9\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\RémiJulien\OneDrive\Documents\DcidConsulting\2.Prestation\3.Formation\3.Production\2025_GRETA DEV IA_P5\OCR_Projet_9\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\RémiJulien\OneDrive\Documents\DcidConsulting\2.Prestation\3.Formation\3.Production\2025_GRETA DEV IA_P5\OCR_Projet_9\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


   HW Add/Mul                → MAE = 1,137.94€
   HW Add/Mul Damped         → MAE = 1,129.08€


c:\Users\RémiJulien\OneDrive\Documents\DcidConsulting\2.Prestation\3.Formation\3.Production\2025_GRETA DEV IA_P5\OCR_Projet_9\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\RémiJulien\OneDrive\Documents\DcidConsulting\2.Prestation\3.Formation\3.Production\2025_GRETA DEV IA_P5\OCR_Projet_9\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


   HW Mul/Mul Damped         → MAE = 1,128.98€

✅ Meilleur HW : HW Add/Add Damped (MAE = 1,119.02€)


---
## 4. SARIMA

In [7]:
# SARIMA avec paramètres manuels
# (p,d,q) = ordre non-saisonnier
# (P,D,Q,s) = ordre saisonnier (s=7 pour hebdomadaire)

print("🔧 Test SARIMA (peut prendre quelques minutes)...")

sarima_configs = [
    {'order': (1, 0, 1), 'seasonal_order': (1, 0, 1, 7), 'name': 'SARIMA(1,0,1)(1,0,1,7)'},
    {'order': (1, 1, 1), 'seasonal_order': (1, 0, 1, 7), 'name': 'SARIMA(1,1,1)(1,0,1,7)'},
    {'order': (2, 1, 2), 'seasonal_order': (1, 0, 1, 7), 'name': 'SARIMA(2,1,2)(1,0,1,7)'},
    {'order': (1, 0, 1), 'seasonal_order': (1, 1, 1, 7), 'name': 'SARIMA(1,0,1)(1,1,1,7)'},
]

best_sarima_mae = float('inf')
best_sarima_pred = None
best_sarima_name = None

for config in sarima_configs:
    try:
        model = SARIMAX(
            ts_train,
            order=config['order'],
            seasonal_order=config['seasonal_order'],
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        fit = model.fit(disp=False, maxiter=200)
        pred = fit.forecast(HORIZON)
        
        mae = mean_absolute_error(test['ca'].values, pred.values)
        print(f"   {config['name']:30} → MAE = {mae:,.2f}€")
        
        if mae < best_sarima_mae:
            best_sarima_mae = mae
            best_sarima_pred = pred.values
            best_sarima_name = config['name']
            best_sarima_model = fit
    except Exception as e:
        print(f"   {config['name']:30} → ❌ Erreur")

if best_sarima_pred is not None:
    print(f"\n✅ Meilleur SARIMA : {best_sarima_name} (MAE = {best_sarima_mae:,.2f}€)")
    results.append(evaluate_model(test['ca'].values, best_sarima_pred, f'SARIMA ({best_sarima_name})'))

🔧 Test SARIMA (peut prendre quelques minutes)...


c:\Users\RémiJulien\OneDrive\Documents\DcidConsulting\2.Prestation\3.Formation\3.Production\2025_GRETA DEV IA_P5\OCR_Projet_9\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\RémiJulien\OneDrive\Documents\DcidConsulting\2.Prestation\3.Formation\3.Production\2025_GRETA DEV IA_P5\OCR_Projet_9\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


   SARIMA(1,0,1)(1,0,1,7)         → MAE = 1,122.39€
   SARIMA(1,1,1)(1,0,1,7)         → MAE = 1,134.62€


c:\Users\RémiJulien\OneDrive\Documents\DcidConsulting\2.Prestation\3.Formation\3.Production\2025_GRETA DEV IA_P5\OCR_Projet_9\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\RémiJulien\OneDrive\Documents\DcidConsulting\2.Prestation\3.Formation\3.Production\2025_GRETA DEV IA_P5\OCR_Projet_9\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\RémiJulien\OneDrive\Documents\DcidConsulting\2.Prestation\3.Formation\3.Production\2025_GRETA DEV IA_P5\OCR_Projet_9\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\RémiJulien\OneDrive\Documents\DcidConsulting\2.Prestation\

   SARIMA(2,1,2)(1,0,1,7)         → MAE = 1,138.28€


c:\Users\RémiJulien\OneDrive\Documents\DcidConsulting\2.Prestation\3.Formation\3.Production\2025_GRETA DEV IA_P5\OCR_Projet_9\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\RémiJulien\OneDrive\Documents\DcidConsulting\2.Prestation\3.Formation\3.Production\2025_GRETA DEV IA_P5\OCR_Projet_9\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


   SARIMA(1,0,1)(1,1,1,7)         → MAE = 1,035.61€

✅ Meilleur SARIMA : SARIMA(1,0,1)(1,1,1,7) (MAE = 1,035.61€)


---
## 5. Régression avec features temporelles

In [8]:
def create_features(df):
    """
    Crée des features temporelles pour la régression.
    """
    df = df.copy()
    df['dayofweek'] = df['date'].dt.dayofweek
    df['dayofmonth'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)
    df['quarter'] = df['date'].dt.quarter
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
    df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
    df['is_month_end'] = df['date'].dt.is_month_end.astype(int)
    
    # Lag features (valeurs passées)
    for lag in [1, 7, 14, 28]:
        df[f'lag_{lag}'] = df['ca'].shift(lag)
    
    # Rolling features (moyennes mobiles)
    for window in [7, 14, 28]:
        df[f'rolling_mean_{window}'] = df['ca'].shift(1).rolling(window=window).mean()
        df[f'rolling_std_{window}'] = df['ca'].shift(1).rolling(window=window).std()
    
    return df

# Créer les features
df_features = create_features(df_daily)

# Supprimer les lignes avec NaN (à cause des lags)
df_features = df_features.dropna()

print(f"✅ {len(df_features)} observations avec features")
print(f"   Features : {df_features.shape[1] - 2} colonnes")

✅ 702 observations avec features
   Features : 18 colonnes


In [9]:
# Split avec features
feature_cols = [c for c in df_features.columns if c not in ['date', 'ca']]

# Attention : on doit recalculer le split car on a perdu des lignes
train_feat = df_features.iloc[:-HORIZON].copy()
test_feat = df_features.iloc[-HORIZON:].copy()

X_train = train_feat[feature_cols]
y_train = train_feat['ca']
X_test = test_feat[feature_cols]
y_test = test_feat['ca']

# Normalisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train : {X_train_scaled.shape}")
print(f"X_test  : {X_test_scaled.shape}")

X_train : (672, 18)
X_test  : (30, 18)


In [10]:
# Tester plusieurs modèles ML
ml_models = [
    ('Ridge', Ridge(alpha=1.0)),
    ('Random Forest', RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)),
    ('Gradient Boosting', GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)),
]

print("🔧 Test des modèles ML...")

best_ml_mae = float('inf')
best_ml_pred = None
best_ml_name = None

for name, model in ml_models:
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    
    mae = mean_absolute_error(y_test.values, pred)
    print(f"   {name:20} → MAE = {mae:,.2f}€")
    
    if mae < best_ml_mae:
        best_ml_mae = mae
        best_ml_pred = pred
        best_ml_name = name
        best_ml_model = model

print(f"\n✅ Meilleur ML : {best_ml_name} (MAE = {best_ml_mae:,.2f}€)")
results.append(evaluate_model(y_test.values, best_ml_pred, f'ML ({best_ml_name})'))

🔧 Test des modèles ML...
   Ridge                → MAE = 1,093.99€
   Random Forest        → MAE = 1,115.15€
   Gradient Boosting    → MAE = 1,254.89€

✅ Meilleur ML : Ridge (MAE = 1,093.99€)


---
## 6. Ensemble (moyenne des meilleurs modèles)

In [11]:
# Combiner les prédictions (moyenne simple)
predictions_to_combine = []

if best_hw_pred is not None:
    predictions_to_combine.append(('HW', best_hw_pred))
if best_sarima_pred is not None:
    predictions_to_combine.append(('SARIMA', best_sarima_pred))
if best_ml_pred is not None:
    predictions_to_combine.append(('ML', best_ml_pred))

if len(predictions_to_combine) >= 2:
    # Moyenne des prédictions
    ensemble_pred = np.mean([p[1] for p in predictions_to_combine], axis=0)
    
    mae_ensemble = mean_absolute_error(test['ca'].values[:len(ensemble_pred)], ensemble_pred)
    print(f"✅ Ensemble ({', '.join([p[0] for p in predictions_to_combine])}) → MAE = {mae_ensemble:,.2f}€")
    
    results.append(evaluate_model(test['ca'].values[:len(ensemble_pred)], ensemble_pred, 'Ensemble'))

✅ Ensemble (HW, SARIMA, ML) → MAE = 1,062.31€


---
## 7. Comparaison des résultats

In [12]:
# Tableau comparatif
df_results = pd.DataFrame(results)
df_results = df_results.sort_values('MAE')

print("\n" + "="*60)
print("📊 COMPARAISON DES MODÈLES")
print("="*60)
print(df_results.to_string(index=False))
print("="*60)

best_model = df_results.iloc[0]
print(f"\n🏆 MEILLEUR MODÈLE : {best_model['model']}")
print(f"   MAE  = {best_model['MAE']:,.2f}€")
print(f"   RMSE = {best_model['RMSE']:,.2f}€")
print(f"   MAPE = {best_model['MAPE']:.2f}%")


📊 COMPARAISON DES MODÈLES
                           model     MAE    RMSE  MAPE
 SARIMA (SARIMA(1,0,1)(1,1,1,7)) 1035.61 1252.04  6.37
                        Ensemble 1062.31 1275.57  6.54
                      ML (Ridge) 1093.99 1339.99  6.66
Holt-Winters (HW Add/Add Damped) 1119.02 1311.89  6.96
                  Baseline (MM7) 1276.03 1450.42  8.01

🏆 MEILLEUR MODÈLE : SARIMA (SARIMA(1,0,1)(1,1,1,7))
   MAE  = 1,035.61€
   RMSE = 1,252.04€
   MAPE = 6.37%


In [13]:
# Visualisation comparaison
fig = px.bar(
    df_results, x='model', y='MAE',
    title='📊 Comparaison des MAE par modèle',
    labels={'model': 'Modèle', 'MAE': 'MAE (€)'},
    color='MAE', color_continuous_scale='RdYlGn_r'
)
fig.update_layout(template='plotly_white', showlegend=False)
fig.show()

In [14]:
# Visualisation des prévisions du meilleur modèle
fig = go.Figure()

# Historique récent
recent = train.tail(60)
fig.add_trace(go.Scatter(
    x=recent['date'], y=recent['ca'],
    name='Historique', line=dict(color='#333')
))

# Réel (test)
fig.add_trace(go.Scatter(
    x=test['date'], y=test['ca'],
    name='Réel', line=dict(color='#FF6B6B', width=2)
))

# Meilleure prédiction
# Identifier le meilleur
best_name = df_results.iloc[0]['model']
if 'Ensemble' in best_name:
    best_pred = ensemble_pred
elif 'ML' in best_name:
    best_pred = best_ml_pred
elif 'SARIMA' in best_name:
    best_pred = best_sarima_pred
else:
    best_pred = best_hw_pred

fig.add_trace(go.Scatter(
    x=test['date'][:len(best_pred)], y=best_pred,
    name=f'Prévision ({best_name})', line=dict(color='#0066FF', width=2, dash='dash')
))

fig.add_vline(x=train['date'].max(), line_dash="dash", line_color="gray")

fig.update_layout(
    title=f'🔮 Prévisions — {best_name}',
    xaxis_title='Date', yaxis_title='CA (€)',
    template='plotly_white', hovermode='x unified'
)
fig.show()

---
## 8. Sauvegarde du meilleur modèle

In [ ]:
import pickle
from pathlib import Path

# Créer le dossier
models_dir = Path('../models/saved')
models_dir.mkdir(parents=True, exist_ok=True)

# Déterminer le meilleur modèle à sauvegarder
best_name = df_results.iloc[0]['model']

if 'ML' in best_name:
    # Sauvegarder modèle ML + scaler + feature_cols
    model_data = {
        'type': 'ml',
        'model': best_ml_model,
        'scaler': scaler,
        'feature_cols': feature_cols,
        'last_date': df_daily['date'].max(),
        'metrics': df_results.iloc[0].to_dict()
    }
    filename = 'best_model_ml.pkl'
elif 'SARIMA' in best_name:
    model_data = {
        'type': 'sarima',
        'model': best_sarima_model,
        'last_date': df_daily['date'].max(),
        'metrics': df_results.iloc[0].to_dict()
    }
    filename = 'best_model_sarima.pkl'
else:
    # Holt-Winters
    model_data = {
        'type': 'holt_winters',
        'model': best_hw_model,
        'last_date': df_daily['date'].max(),
        'metrics': df_results.iloc[0].to_dict()
    }
    filename = 'best_model_hw.pkl'

model_path = models_dir / filename
with open(model_path, 'wb') as f:
    pickle.dump(model_data, f)

print(f"✅ Modèle sauvegardé : {model_path}")
print(f"   Type : {model_data['type']}")
print(f"   MAE  : {model_data['metrics']['MAE']}€")

---
## 📝 Résumé

### Modèles testés

| Modèle | Description |
|--------|-------------|
| Baseline | Moyenne mobile 7 jours |
| Holt-Winters | 5 configurations testées |
| SARIMA | 4 configurations testées |
| ML | Ridge, Random Forest, Gradient Boosting |
| Ensemble | Moyenne des meilleurs |

### Conclusion

Le meilleur modèle a été identifié et sauvegardé pour l'intégration API.